# Day 7: Data Cleaning & Visualization
MLB Internship Curriculum

This Google Colab notebook performs Data Cleaning, Data Visualization using Matplotlib & Seaborn, and deploys an interactive Gradio Dashboard with Ngrok integration.

In [ ]:
# Install dependencies for Google Colab
!pip install pandas matplotlib seaborn gradio pyngrok --quiet

In [ ]:
import os
import pandas as pd
import numpy as np

csv_path = 'student_performance.csv'
if not os.path.exists(csv_path):
    csv_data = '''Student_ID,Name,Age,Program,Python,Mathematics,Statistics,Machine_Learning,Attendance
S001,Ali Khan,20,AI,85,78,92,88,95
S002,Sara Ahmed,21,AI,72,75,70,80,90
S003,Ahmed Raza,22,SE,90,88,91,93,96
S004,Fatima Noor,20,DS,65,70,68,72,85
S005,Usman Ali,21,AI,78,82,80,76,88
S006,Ayesha Malik,22,SE,95,94,96,97,99
S007,Hassan Tariq,20,DS,55,60,58,62,75
S008,Zainab Iqbal,21,AI,88,86,90,91,94
S009,Bilal Ahmed,23,SE,73,77,75,79,82
S010,Maryam Khan,20,DS,81,84,79,85,91
S011,Hamza Siddiqui,22,AI,69,71,74,70,87
S012,Noor Fatima,21,SE,92,90,93,95,98
S013,Talha Javed,20,DS,76,74,78,80,89
S014,Iqra Aslam,22,AI,84,83,86,88,92
S015,Danish Ali,23,SE,61,65,63,67,78
S016,Hira Shah,21,DS,89,91,88,90,96
S017,Omar Farooq,20,AI,74,72,76,78,84
S018,Laiba Khan,22,SE,97,95,98,99,100
S019,Abdullah,21,DS,68,66,70,72,80
S020,Mehwish,20,AI,86,89,87,90,93'''
    with open(csv_path, 'w') as f:
        f.write(csv_data.strip())

df = pd.read_csv(csv_path)
print('Missing Values:\n', df.isna().sum())
df = df.drop_duplicates()
df.columns = [col.strip().replace(' ', '_') for col in df.columns]

subjects = ['Python', 'Mathematics', 'Statistics', 'Machine_Learning']
for col in subjects:
    df[col] = pd.to_numeric(df[col], errors='coerce')

def assign_performance(score):
    if score >= 90: return 'Excellent'
    elif score >= 80: return 'Good'
    elif score >= 70: return 'Average'
    else: return 'Needs Improvement'

df['Average_Score'] = df[subjects].mean(axis=1).round(2)
df['Performance'] = df['Average_Score'].apply(assign_performance)
df = df.sort_values(by='Average_Score', ascending=False)
df.to_csv('cleaned_student_performance.csv', index=False)
print('Cleaned dataset saved as cleaned_student_performance.csv')
df.head()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Bar Chart
plt.figure(figsize=(10, 5))
sns.barplot(x='Name', y='Average_Score', data=df, hue='Name', legend=False)
plt.xticks(rotation=45, ha='right')
plt.title('Average Score per Student')
plt.tight_layout()
plt.savefig('bar_chart.png')
plt.show()

# 2. Histogram
plt.figure(figsize=(8, 5))
sns.histplot(df['Average_Score'], kde=True, bins=8)
plt.title('Average Score Distribution')
plt.tight_layout()
plt.savefig('histogram.png')
plt.show()

# 3. Scatter Plot
plt.figure(figsize=(8, 5))
sns.scatterplot(x='Python', y='Machine_Learning', hue='Performance', style='Program', s=100, data=df)
plt.title('Python vs Machine Learning Marks')
plt.tight_layout()
plt.savefig('scatter_plot.png')
plt.show()

# 4. Pie Chart
plt.figure(figsize=(6, 6))
counts = df['Performance'].value_counts()
plt.pie(counts, labels=counts.index, autopct='%1.1f%%', startangle=140)
plt.title('Performance Categories Distribution')
plt.tight_layout()
plt.savefig('pie_chart.png')
plt.show()

# 5. Box Plot
plt.figure(figsize=(8, 5))
sns.boxplot(data=df[subjects])
plt.title('Marks Distribution Across All Subjects')
plt.tight_layout()
plt.savefig('box_plot.png')
plt.show()

In [ ]:
# Launch Gradio App directly in Google Colab with public sharing link
import gradio as gr
import matplotlib.pyplot as plt
import seaborn as sns

subjects = ['Python', 'Mathematics', 'Statistics', 'Machine_Learning']

def get_summary():
    total = len(df)
    subj_means = df[subjects].mean().round(2)
    highest_subj = f"{subj_means.idxmax()} ({subj_means.max()})"
    needs_imp = len(df[df['Performance'] == 'Needs Improvement'])
    return f"Total Students: {total}\nHighest Average Subject: {highest_subj}\nStudents Needing Improvement: {needs_imp}"

def filter_students(performance_filter, program_filter):
    filtered = df.copy()
    if performance_filter != 'All':
        filtered = filtered[filtered['Performance'] == performance_filter]
    if program_filter != 'All':
        filtered = filtered[filtered['Program'] == program_filter]
    return filtered[['Student_ID', 'Name', 'Program', 'Average_Score', 'Performance']]

def generate_chart(chart_type):
    fig, ax = plt.subplots(figsize=(8, 4))
    if chart_type == 'Bar Chart (Average Score per Student)':
        sns.barplot(x='Name', y='Average_Score', data=df, ax=ax, hue='Name', legend=False)
        plt.xticks(rotation=45, ha='right')
        ax.set_title('Average Score per Student')
    elif chart_type == 'Histogram (Score Distribution)':
        sns.histplot(df['Average_Score'], kde=True, bins=8, ax=ax)
        ax.set_title('Score Distribution')
    elif chart_type == 'Scatter Plot (Python vs ML)':
        sns.scatterplot(x='Python', y='Machine_Learning', hue='Performance', data=df, ax=ax, s=90)
        ax.set_title('Python vs Machine Learning Marks')
    elif chart_type == 'Pie Chart (Performance Breakdown)':
        counts = df['Performance'].value_counts()
        ax.pie(counts, labels=counts.index, autopct='%1.1f%%', startangle=140)
        ax.set_title('Performance Categories Breakdown')
    elif chart_type == 'Box Plot (Marks Across Subjects)':
        sns.boxplot(data=df[subjects], ax=ax)
        ax.set_title('Marks Across Core Subjects')
    plt.tight_layout()
    return fig

with gr.Blocks(title='Student Performance Dashboard') as app:
    gr.Markdown('# Student Performance Dashboard')
    with gr.Row():
        summary_box = gr.Textbox(label='Summary Metrics', value=get_summary(), interactive=False)
    with gr.Row():
        perf_dropdown = gr.Dropdown(choices=['All', 'Excellent', 'Good', 'Average', 'Needs Improvement'], value='All', label='Filter by Performance')
        prog_dropdown = gr.Dropdown(choices=['All', 'AI', 'DS', 'SE'], value='All', label='Filter by Program')
    table_output = gr.Dataframe(value=df[['Student_ID', 'Name', 'Program', 'Average_Score', 'Performance']], label='Student Data')
    perf_dropdown.change(filter_students, inputs=[perf_dropdown, prog_dropdown], outputs=table_output)
    prog_dropdown.change(filter_students, inputs=[perf_dropdown, prog_dropdown], outputs=table_output)
    gr.Markdown('## Visualizations')
    chart_dropdown = gr.Dropdown(choices=['Bar Chart (Average Score per Student)', 'Histogram (Score Distribution)', 'Scatter Plot (Python vs ML)', 'Pie Chart (Performance Breakdown)', 'Box Plot (Marks Across Subjects)'], value='Bar Chart (Average Score per Student)', label='Select Chart')
    plot_output = gr.Plot(value=generate_chart('Bar Chart (Average Score per Student)'))
    chart_dropdown.change(generate_chart, inputs=chart_dropdown, outputs=plot_output)

app.launch(share=True)

In [ ]:
# Optional: Launch Ngrok Tunnel for Gradio / Local Server port 7860
from pyngrok import ngrok
import time

# Set ngrok token if needed:
# ngrok.set_auth_token('YOUR_NGROK_TOKEN')
try:
    public_url = ngrok.connect(7860)
    print('Ngrok Public Tunnel URL:', public_url)
except Exception as e:
    print('Ngrok info:', e)